# Concerts

The central data set is the list of concerts assembled over many years of work by a San Antonio Public Library librarian, Andy Crews. It was passed on to two UTSA librarians, William Glenn and Jacob Sherman, and from there to Diane Lopez. Diane Lopez modified the file by splitting events at which many artists played into multiple lines, one for each artist.

This data is in the master data file, in a sheet called `Venue_Artist_Event` and will be presented as is, with no modifications. I will just add the unique venue and artist ids corresponding to each venue and artist. For clarity, I will then rename "Index" as "concertIndex" to distinguish it from artist and venue indexes.

In [1]:
import pandas as pd
import os

In [2]:
concerts = pd.read_excel('input-data/product1_data_master.xlsx', sheet_name='Venue_Artist_Event')

In [3]:
concerts.info()

<class 'pandas.DataFrame'>
RangeIndex: 37892 entries, 0 to 37891
Data columns (total 8 columns):
 #   Column          Non-Null Count  Dtype
---  ------          --------------  -----
 0   Month           37892 non-null  str  
 1   Day             37892 non-null  int64
 2   Year            37892 non-null  int64
 3   Venue           37892 non-null  str  
 4   Event_Artists   37892 non-null  str  
 5   Artist_Formula  37892 non-null  str  
 6   Event_Formulas  7723 non-null   str  
 7   Index           37892 non-null  int64
dtypes: int64(3), str(5)
memory usage: 2.3 MB


Now I need the artist and venue ids, so I'll load the corresponding data frames.

In [4]:
artists = pd.read_excel('output-data/sounds-of-san-anto.xlsx', sheet_name='artists')
venues = pd.read_excel('output-data/sounds-of-san-anto.xlsx', sheet_name='venues')

## Add Artist Index

Now, I'll inspect the artists data frame. Then I create a lookup data frame with artists and their indexes. I add it to the concerts data frame. Unfortunately, it leaves 34 rows with no artist index. 

In [5]:
artists.info()

<class 'pandas.DataFrame'>
RangeIndex: 20320 entries, 0 to 20319
Data columns (total 4 columns):
 #   Column            Non-Null Count  Dtype
---  ------            --------------  -----
 0   Index             20320 non-null  int64
 1   Artist            20320 non-null  str  
 2   First Appearance  20320 non-null  str  
 3   Last Appearance   20320 non-null  str  
dtypes: int64(1), str(3)
memory usage: 635.1 KB


In [6]:
artist_index_lookup = artists.set_index("Artist")["Index"]
concerts["artistIndex"] = concerts["Artist_Formula"].map(artist_index_lookup)

In [7]:
concerts.info()

<class 'pandas.DataFrame'>
RangeIndex: 37892 entries, 0 to 37891
Data columns (total 9 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   Month           37892 non-null  str    
 1   Day             37892 non-null  int64  
 2   Year            37892 non-null  int64  
 3   Venue           37892 non-null  str    
 4   Event_Artists   37892 non-null  str    
 5   Artist_Formula  37892 non-null  str    
 6   Event_Formulas  7723 non-null   str    
 7   Index           37892 non-null  int64  
 8   artistIndex     37858 non-null  float64
dtypes: float64(1), int64(3), str(5)
memory usage: 2.6 MB


In [8]:
# How many rows failed to match?
unmatched = concerts["artistIndex"].isna().sum()
print(f"{unmatched} unmatched rows out of {len(concerts)}")

34 unmatched rows out of 37892


I inspect the list of concerts that didn't get an artist index. It doesn't look like the problem is to do with case sensitivity, because Porter Waggoner for example is in both the artists and concerts lists. I suspect the problem is different trailing whitespaces.

In [9]:
concerts[concerts["artistIndex"].isna()]

,Month,Day,Year,Venue,Event_Artists,Artist_Formula,Event_Formulas,Index,artistIndex
4860,February,16,1970,Freeman Coliseum,San Antonio Stock Show & Rodeo: Porter Waggone...,Porter Waggoner,San Antonio Stock Show & Rodeo,2591,NaN
4861,February,17,1970,Freeman Coliseum,San Antonio Stock Show & Rodeo: Porter Waggone...,Porter Waggoner,San Antonio Stock Show & Rodeo,2592,NaN
4862,February,18,1970,Freeman Coliseum,San Antonio Stock Show & Rodeo: Porter Waggone...,Porter Waggoner,San Antonio Stock Show & Rodeo,2593,NaN
4863,February,19,1970,Freeman Coliseum,San Antonio Stock Show & Rodeo: Porter Waggone...,Porter Waggoner,San Antonio Stock Show & Rodeo,2594,NaN
4865,February,20,1970,Freeman Coliseum,"San Antonio Stock Show & Rodeo: George Jones, ...",George Jones,San Antonio Stock Show & Rodeo,2596,NaN
4943,June,26,1970,Randy's,Waylon Jennings,Waylon Jennings,NaN,2646,NaN
5004,September,13,1970,JAM Factory,Leon Russell,Leon Russell,NaN,2682,NaN
5048,December,3,1970,Lila Cockrell Theatre,"San Antonio Symphony Orchestra, Robert Casades...",San Antonio Symphony Orchestra,NaN,2710,NaN
5051,December,5,1970,Lila Cockrell Theatre,"San Antonio Symphony Orchestra, Robert Casades...",Robert Casadesus (piano),NaN,2711,NaN
5122,February,19,1971,Freeman Coliseum,San Antonio Stock Show and Rodeo: Lynn Anderso...,Lynn Anderson,San Antonio Stock Show and Rodeo,2759,NaN


First I create two temporary columns containing artist names stripped of extra whitespace. When I try to use these columns to identify the missing artist indexes, I run into a problem. There are duplicate artists in this column, which is leading to duplicate indexes, which is causing the code to fail. I inspect this problem and find three artists that have duplicate entries distinguished only by differences in white space. These should be cleaned up at some point, but right now I am not working on data cleanup, so I leave it be.

In [10]:
# Normalize whitespace on both sides (leading/trailing, and collapse internal double-spaces too)
artists["_Artist_clean"] = artists["Artist"].str.strip().str.replace(r"\s+", " ", regex=True)
concerts["_Artist_Formula_clean"] = concerts["Artist_Formula"].str.strip().str.replace(r"\s+", " ", regex=True)

dupes = artists[artists["_Artist_clean"].duplicated(keep=False)].sort_values("_Artist_clean")
print(dupes[["Index", "Artist", "_Artist_clean"]])

      Index                           Artist                    _Artist_clean
696    2378    Angel Flores y  Los Alacranes     Angel Flores y Los Alacranes
697    6881     Angel Flores y Los Alacranes     Angel Flores y Los Alacranes
4285   9417  David Lee Garza y Los Musicales  David Lee Garza y Los Musicales
4286   3621  David Lee Garza y Los Musicales  David Lee Garza y Los Musicales
7680   2804            It's  A Beautiful Day             It's A Beautiful Day
7681   4018             It's A Beautiful Day             It's A Beautiful Day


I can see that none of these are in the list of artists that are missing indexes. In order to move past this hiccup, I temporarily drop one of each pair. Now I can create a lookup table with the artist indexes and the clean, no-whitespace artist names, since I no longer have the problem of duplicate indexes. Then, I find the rows that are missing indexes and map the indexes from the cleaned column. When this process is complete, I see that there are no longer any null artist indexes. 

In [11]:
# Drop duplicate clean names, keeping first — safe since none of these are the ones we need
artists_deduped = artists.drop_duplicates(subset="_Artist_clean", keep="first")

artist_to_index_clean = artists_deduped.set_index("_Artist_clean")["Index"]

mask = concerts["artistIndex"].isna()
concerts.loc[mask, "artistIndex"] = concerts.loc[mask, "_Artist_Formula_clean"].map(artist_to_index_clean)

print(f"{concerts['artistIndex'].isna().sum()} unmatched after whitespace cleanup")

0 unmatched after whitespace cleanup


Now I can clean up the temporary columns I created in each data frame. I also need to convert the indexes to integer values. They had all become floats because there were null values, and NaN is a floating point value, so whenver there is NaN in a column, the column is float data type. Here I convert it back and double check the results.

In [12]:
# Drop the helper columns you no longer need
concerts.drop(columns="_Artist_Formula_clean", inplace=True)
artists.drop(columns="_Artist_clean", inplace=True)

# Convert artistIndex to nullable integer (it's currently float because of the earlier NaNs)
concerts["artistIndex"] = concerts["artistIndex"].astype("Int64")

# Sanity check: confirm no nulls remain and dtype is right
print(concerts["artistIndex"].isna().sum())
print(concerts["artistIndex"].dtype)

0
Int64


In [13]:
concerts.info()

<class 'pandas.DataFrame'>
RangeIndex: 37892 entries, 0 to 37891
Data columns (total 9 columns):
 #   Column          Non-Null Count  Dtype
---  ------          --------------  -----
 0   Month           37892 non-null  str  
 1   Day             37892 non-null  int64
 2   Year            37892 non-null  int64
 3   Venue           37892 non-null  str  
 4   Event_Artists   37892 non-null  str  
 5   Artist_Formula  37892 non-null  str  
 6   Event_Formulas  7723 non-null   str  
 7   Index           37892 non-null  int64
 8   artistIndex     37892 non-null  Int64
dtypes: Int64(1), int64(3), str(5)
memory usage: 2.6 MB


Sure enough, all the artists have indexes, so I'm ready to move on to the venue Indexes.

## Add Venue Indexes

I inspect the venues data frame and follow the same procedure as for artists. I create a lookup data frame and then map all the indexes based on that data frame. When I'm done, I find seven rows have no venue indexes. I investigate the problem and find that the concerts at South Texas Popular Culture Center didn't get a venue index. Probably there is a difference in trailing or leading whitespace in those rows.

In [14]:
venues.info()

<class 'pandas.DataFrame'>
RangeIndex: 2971 entries, 0 to 2970
Data columns (total 4 columns):
 #   Column            Non-Null Count  Dtype
---  ------            --------------  -----
 0   Index             2971 non-null   int64
 1   Venue             2971 non-null   str  
 2   First Appearance  2602 non-null   str  
 3   Last Appearance   2602 non-null   str  
dtypes: int64(1), str(3)
memory usage: 93.0 KB


In [15]:
venue_index_lookup = venues.set_index("Venue")["Index"]
concerts["venueIndex"] = concerts["Venue"].map(venue_index_lookup)
concerts.info()

<class 'pandas.DataFrame'>
RangeIndex: 37892 entries, 0 to 37891
Data columns (total 10 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   Month           37892 non-null  str    
 1   Day             37892 non-null  int64  
 2   Year            37892 non-null  int64  
 3   Venue           37892 non-null  str    
 4   Event_Artists   37892 non-null  str    
 5   Artist_Formula  37892 non-null  str    
 6   Event_Formulas  7723 non-null   str    
 7   Index           37892 non-null  int64  
 8   artistIndex     37892 non-null  Int64  
 9   venueIndex      37886 non-null  float64
dtypes: Int64(1), float64(1), int64(3), str(5)
memory usage: 2.9 MB


In [16]:
concerts[concerts['venueIndex'].isna()]

,Month,Day,Year,Venue,Event_Artists,Artist_Formula,Event_Formulas,Index,artistIndex,venueIndex
28028,September,1,2013,South Texas Popular Culture Center,"Texas Legacy Music Concert: Augie Meyers, Dell...",Augie Meyers,Texas Legacy Music Concert,16271,784,NaN
28029,September,1,2013,South Texas Popular Culture Center,"Texas Legacy Music Concert: Augie Meyers, Dell...",Dell-Kings,Texas Legacy Music Concert,16271,9539,NaN
28030,September,1,2013,South Texas Popular Culture Center,"Texas Legacy Music Concert: Augie Meyers, Dell...",The Krayolas,Texas Legacy Music Concert,16271,663,NaN
28031,September,1,2013,South Texas Popular Culture Center,"Texas Legacy Music Concert: Augie Meyers, Dell...",The West Side Horns,Texas Legacy Music Concert,16271,2374,NaN
28032,September,1,2013,South Texas Popular Culture Center,"Texas Legacy Music Concert: Augie Meyers, Dell...",Larry Lange with Ernie Garibay and Joe Jama,Texas Legacy Music Concert,16271,13308,NaN
28033,September,1,2013,South Texas Popular Culture Center,"Texas Legacy Music Concert: Augie Meyers, Dell...",the Teen Canteen All Stars,Texas Legacy Music Concert,16271,20412,NaN


It's only one venue, so I'm going to fix it manually. I create a temporary data frame in which all the venues have the white space normalized. Then I find all the rows that have South Texas Popular Culture Center in this data frame. There are actually seven rows -- one of them must have had the same exact white space as in the venues list, and so got matched, while the other six didn't. 

I see that in fact South Texas Popular Culture Center appears twice in the venues list with different white spaces. Since I'm not fixing or cleaning data right now, I leave that issue alone and just add the 2530 index to the ones that didn't match.

In [20]:
# Normalize whitespace on the Venue column the same way you did before
venue_clean = concerts["Venue"].str.strip().str.replace(r"\s+", " ", regex=True)

# Build the mask by comparing normalized venue name
mask = venue_clean == "South Texas Popular Culture Center"

print(f"{mask.sum()} matching rows found")

7 matching rows found


In [22]:
concerts.loc[mask & concerts["venueIndex"].isna(), "venueIndex"] = 2530
concerts.info()

<class 'pandas.DataFrame'>
RangeIndex: 37892 entries, 0 to 37891
Data columns (total 10 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   Month           37892 non-null  str    
 1   Day             37892 non-null  int64  
 2   Year            37892 non-null  int64  
 3   Venue           37892 non-null  str    
 4   Event_Artists   37892 non-null  str    
 5   Artist_Formula  37892 non-null  str    
 6   Event_Formulas  7723 non-null   str    
 7   Index           37892 non-null  int64  
 8   artistIndex     37892 non-null  Int64  
 9   venueIndex      37892 non-null  float64
dtypes: Int64(1), float64(1), int64(3), str(5)
memory usage: 2.9 MB


I just want to rename the Index column so it's clear that refers to concerts.

In [24]:
concerts = concerts.rename(columns={'Index': 'concertIndex'})
concerts.info()

<class 'pandas.DataFrame'>
RangeIndex: 37892 entries, 0 to 37891
Data columns (total 10 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   Month           37892 non-null  str    
 1   Day             37892 non-null  int64  
 2   Year            37892 non-null  int64  
 3   Venue           37892 non-null  str    
 4   Event_Artists   37892 non-null  str    
 5   Artist_Formula  37892 non-null  str    
 6   Event_Formulas  7723 non-null   str    
 7   concertIndex    37892 non-null  int64  
 8   artistIndex     37892 non-null  Int64  
 9   venueIndex      37892 non-null  float64
dtypes: Int64(1), float64(1), int64(3), str(5)
memory usage: 2.9 MB


## Write to Excel

Now I am ready to write the frame to Excel.

In [25]:
def write_df_to_excel(df, filepath, sheet_name):
    """Write a DataFrame to a sheet in an Excel file, creating the file
    if it doesn't exist, or replacing the sheet if it does."""
    
    mode = "a" if os.path.exists(filepath) else "w"
    kwargs = {"if_sheet_exists": "replace"} if mode == "a" else {}

    with pd.ExcelWriter(filepath, engine="openpyxl", datetime_format='YYYY-MM-DD', mode=mode, **kwargs) as writer:
        df.to_excel(writer, sheet_name=sheet_name, index=False)

In [27]:
# write_df_to_excel(concerts, 'output-data/sounds-of-san-anto.xlsx', 'concerts')